# 과제 2 - 분류
### churn_train.csv 를 이용하여 churn_test.csv 의 churn 을 분류

#### churn_dataspec(데이터명세서) 참고
- 타깃(클래스)
- Churn : 1=이탈, 0=유지
- 특징(입력 변수)
- Age : 나이
- Tenure_Months : 가입기간(개월)
- Monthly_Fee : 월 요금
- Total_Usage_GB : 최근 사용량(GB)
- Support_Tickets_3M : 최근 3개월 CS 문의 건수
- Late_Payments_6M : 최근 6개월 연체 횟수
- Contract : 계약 형태(Month-to-month / 1-year / 2-year)
- AutoPay : 자동결제(0/1)
- Internet_Type : 회선(Fiber/DSL/5G)
- Has_Addon : 부가서비스(0/1)
- NPS_Score : 만족도 점수(-100~100)
- Region : 지역(Seoul/Metro/Other)

In [ ]:
# 커널 확인
print("hello")

hello


In [2]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler


In [ ]:
# 데이터 불러오기
train = pd.read_csv("./data/churn_train.csv")
test = pd.read_csv("./data/churn_test.csv")
target = "Churn"
id_col = "Customer_ID"

In [11]:
train.head()

,Customer_ID,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,22216,38,1,70096,449,1,0,Month-to-month,1,DSL,1,-57,Seoul,0
1,22583,33,11,38262,564,2,1,Month-to-month,0,Fiber,0,69,Seoul,1
2,21663,27,34,62086,362,1,0,Month-to-month,1,Fiber,1,21,Metro,1
3,23028,56,19,72615,595,0,0,1-year,0,DSL,0,16,Seoul,0
4,24344,31,62,63279,365,2,1,2-year,0,5G,0,13,Metro,0


In [12]:
test.head()

,Customer_ID,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,26891,25,6,62606,450,0,1,Month-to-month,0,Fiber,0,12,Seoul,0
1,27712,29,8,57140,437,0,1,2-year,0,DSL,0,-9,Metro,0
2,25001,28,17,68910,466,2,1,Month-to-month,0,Fiber,0,24,Seoul,0
3,25854,53,23,64286,494,2,0,Month-to-month,1,Fiber,1,72,Metro,0
4,21280,63,12,62359,129,0,0,Month-to-month,1,5G,0,-41,Other,0


In [ ]:
# 수치형 범주형 컬럼 분리
num_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_cols.remove(id_col)  # 식별자 컬럼 제외
num_cols.remove(target)  # 타깃 컬럼 제외
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print("수치형 컬럼:", num_cols)
print("범주형 컬럼:", cat_cols)


수치형 컬럼: ['Age', 'Tenure_Months', 'Monthly_Fee', 'Total_Usage_GB', 'Support_Tickets_3M', 'Late_Payments_6M', 'AutoPay', 'Has_Addon', 'NPS_Score']
범주형 컬럼: ['Contract', 'Internet_Type', 'Region']


/var/folders/tv/g7ngq9yd0knchh30z08jp4yh0000gn/T/ipykernel_85112/211314437.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
# EDA
print("클래스 분포:")
print(train[target].value_counts())
print("\n결측치 개수:")
print(train.isnull().sum())

클래스 분포:
Churn
0    4739
1    1661
Name: count, dtype: int64

결측치 개수:
Customer_ID           0
Age                   0
Tenure_Months         0
Monthly_Fee           0
Total_Usage_GB        0
Support_Tickets_3M    0
Late_Payments_6M      0
Contract              0
AutoPay               0
Internet_Type         0
Has_Addon             0
NPS_Score             0
Region                0
Churn                 0
dtype: int64


In [16]:
print(train[num_cols].describe())

               Age  Tenure_Months    Monthly_Fee  Total_Usage_GB  \
count  6400.000000    6400.000000    6400.000000     6400.000000   
mean     43.295313      20.120781   68503.549844      378.990781   
std      15.037027      13.974413   11963.755134      140.174917   
min      18.000000       1.000000   35000.000000       30.000000   
25%      30.000000      10.000000   60485.500000      284.000000   
50%      43.000000      17.000000   68433.000000      379.000000   
75%      56.000000      27.000000   76443.750000      474.000000   
max      69.000000      72.000000  110676.000000      900.000000   

       Support_Tickets_3M  Late_Payments_6M      AutoPay    Has_Addon  \
count         6400.000000       6400.000000  6400.000000  6400.000000   
mean             0.817500          0.510938     0.553281     0.446562   
std              0.900143          0.700192     0.497192     0.497175   
min              0.000000          0.000000     0.000000     0.000000   
25%              0.000

In [ ]:
# 피처/타깃 분리
X_train = train.drop(columns=[id_col, target]) # Customer_ID는 모델 입력에서 제외
y_train = train[target] # Churn을 y로 분리
X_test = test.drop(columns=[id_col])
X_test.head()

,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,25,6,62606,450,0,1,Month-to-month,0,Fiber,0,12,Seoul,0
1,29,8,57140,437,0,1,2-year,0,DSL,0,-9,Metro,0
2,28,17,68910,466,2,1,Month-to-month,0,Fiber,0,24,Seoul,0
3,53,23,64286,494,2,0,Month-to-month,1,Fiber,1,72,Metro,0
4,63,12,62359,129,0,0,Month-to-month,1,5G,0,-41,Other,0


In [ ]:
# 전처리 ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)


In [ ]:
# 모델 비교: Logistic Regression, RandomForestClassifier, GradientBoosting
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
}
for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")
    print(f"{name} - Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})") # 베이스라인 모델 비교 결과 출력


Logistic Regression - Accuracy: 0.7614 (+/- 0.0081)
Random Forest - Accuracy: 0.7488 (+/- 0.0091)
Gradient Boosting - Accuracy: 0.7583 (+/- 0.0081)


In [ ]:
# 모델 선택
from sklearn.model_selection import GridSearchCV
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10],
}
pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", RandomForestClassifier())])
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)


Best parameters: {'model__max_depth': 10, 'model__n_estimators': 100}
Best CV Accuracy: 0.7584375


In [ ]:
# 최종 학습
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

# 테스트 예측
test_predictions = best_model.predict(X_test)
submission = pd.DataFrame({id_col: test[id_col], target: test_predictions})

In [ ]:
# 최종 Accuracy, confusion matrix, classification_report를 확인
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_test = test[target]
accuracy = accuracy_score(y_test, test_predictions)
conf_matrix = confusion_matrix(y_test, test_predictions)
class_report = classification_report(y_test, test_predictions)
print(f"Test Accuracy: {accuracy:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

Test Accuracy: 0.7881
Confusion Matrix:
[[1187   38]
 [ 301   74]]
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.97      0.88      1225
           1       0.66      0.20      0.30       375

    accuracy                           0.79      1600
   macro avg       0.73      0.58      0.59      1600
weighted avg       0.77      0.79      0.74      1600

